# Test Retrieval With BGE Reranker

This notebook compares the original embedding-only retrieval with the new two-stage pipeline:

1. **Candidate retrieval** with `BAAI/bge-small-en-v1.5` embeddings and cosine similarity.
2. **Reranking** with `BAAI/bge-reranker-base`, which scores each `[query, article]` pair directly.

The first run may take longer because Hugging Face models need to be downloaded or loaded from cache.

In [1]:
import pandas as pd

from retriever import (
    compare_retrieval_with_and_without_rerank,
    get_articles_from_query,
)

## Choose a query

Pick a news topic that appears across multiple dates or years. Smaller `candidate_k` is faster; larger `candidate_k` gives the reranker more candidates to improve.

In [6]:
query = "Covid-19 pandemic impact in healthcare"
top_k = 10
candidate_k = 50

## Compare embedding-only vs reranked retrieval

In [7]:
embedding_only, reranked = compare_retrieval_with_and_without_rerank(
    query,
    top_k=top_k,
    candidate_k=candidate_k,
)

display_cols = ["date", "title", "embedding_score", "rerank_score", "url"]

print("Embedding-only results")
display(embedding_only[display_cols])

print("Reranked results")
display(reranked[display_cols])

Embedding-only results


,date,title,embedding_score,rerank_score,url
0,2020-08-06,Small boats and female workers hardest hit by ...,0.813371,0.813371,https://www.theguardian.com/environment/2020/j...
1,2020-11-05,Women are on the Covid-19 frontline – we must ...,0.790953,0.790953,https://www.theguardian.com/global-development...
2,2021-06-01,UK BAME people: how has the coronavirus pandem...,0.823788,0.823788,https://www.theguardian.com/world/2021/jan/06/...
3,2021-07-06,"Spat at, abused, attacked: healthcare staff fa...",0.819246,0.819246,https://www.theguardian.com/global-development...
4,2022-02-03,"Covid has intensified gender inequalities, glo...",0.776233,0.776233,https://www.theguardian.com/world/2022/mar/02/...
5,2022-09-04,"As Britain learns to live with Covid, it faces...",0.777463,0.777463,https://www.theguardian.com/world/2022/apr/09/...
6,2023-01-11,Covid pandemic ‘had lasting impact’ on brain h...,0.809899,0.809899,https://www.theguardian.com/society/2023/nov/0...
7,2023-07-04,‘A silver lining’: how Covid ushered in a vacc...,0.778483,0.778483,https://www.theguardian.com/science/2023/apr/0...
8,2025-02-01,Are we ready for another pandemic?,0.776435,0.776435,https://www.theguardian.com/global-development...
9,2025-07-03,People in the US: tell us how your life has ch...,0.771202,0.771202,https://www.theguardian.com/world/2025/mar/07/...


Reranked results


,date,title,embedding_score,rerank_score,url
0,2020-07-06,Lockdown and the impact on deaf children | Letter,0.775287,5.083216,https://www.theguardian.com/society/2020/jun/0...
1,2020-12-12,'Lonely repetition and growing nihilism': how ...,0.782856,5.011456,https://www.theguardian.com/australia-news/202...
2,2021-07-06,"Spat at, abused, attacked: healthcare staff fa...",0.819246,3.634744,https://www.theguardian.com/global-development...
3,2021-11-02,Covid-19 has put a major strain on mental heal...,0.772680,4.053048,https://www.theguardian.com/society/2021/feb/1...
4,2022-03-02,Australian women thrust into economic insecuri...,0.773231,-0.874560,https://www.theguardian.com/australia-news/202...
5,2022-09-04,"As Britain learns to live with Covid, it faces...",0.777463,2.473818,https://www.theguardian.com/world/2022/apr/09/...
6,2023-01-11,Covid pandemic ‘had lasting impact’ on brain h...,0.809899,2.793876,https://www.theguardian.com/society/2023/nov/0...
7,2023-08-03,Covid’s effect on mental health not as great a...,0.776550,0.808682,https://www.theguardian.com/society/2023/mar/0...
8,2025-02-01,Are we ready for another pandemic?,0.776435,0.771274,https://www.theguardian.com/global-development...
9,2025-09-03,‘The pandemic reinforced existing inequalities...,0.770412,0.297891,https://www.theguardian.com/world/2025/mar/09/...


## Inspect year coverage

The final retriever still keeps the timeline-aware behavior: after reranking, selected articles are diversified by year and sorted chronologically.

In [8]:
def add_year(df):
    df = df.copy()
    df["year"] = pd.to_datetime(df["date"], errors="coerce").dt.year
    return df

year_compare = pd.DataFrame({
    "embedding_only_years": [sorted(add_year(embedding_only)["year"].dropna().unique().tolist())],
    "reranked_years": [sorted(add_year(reranked)["year"].dropna().unique().tolist())],
})

display(year_compare)

,embedding_only_years,reranked_years
0,"[2020, 2021, 2022, 2023, 2025]","[2020, 2021, 2022, 2023, 2025]"


## Use the final retrieval function

This is the function used by the app. Set `use_reranker=False` if you want the old behavior.

In [9]:
articles = get_articles_from_query(
    query,
    top_k=top_k,
    candidate_k=candidate_k,
    use_reranker=True,
)

pd.DataFrame(articles)[display_cols]

,date,title,embedding_score,rerank_score,url
0,2020-07-06,Lockdown and the impact on deaf children | Letter,0.775287,5.083216,https://www.theguardian.com/society/2020/jun/0...
1,2020-12-12,'Lonely repetition and growing nihilism': how ...,0.782856,5.011456,https://www.theguardian.com/australia-news/202...
2,2021-07-06,"Spat at, abused, attacked: healthcare staff fa...",0.819246,3.634744,https://www.theguardian.com/global-development...
3,2021-11-02,Covid-19 has put a major strain on mental heal...,0.772680,4.053048,https://www.theguardian.com/society/2021/feb/1...
4,2022-03-02,Australian women thrust into economic insecuri...,0.773231,-0.874560,https://www.theguardian.com/australia-news/202...
5,2022-09-04,"As Britain learns to live with Covid, it faces...",0.777463,2.473818,https://www.theguardian.com/world/2022/apr/09/...
6,2023-01-11,Covid pandemic ‘had lasting impact’ on brain h...,0.809899,2.793876,https://www.theguardian.com/society/2023/nov/0...
7,2023-08-03,Covid’s effect on mental health not as great a...,0.776550,0.808682,https://www.theguardian.com/society/2023/mar/0...
8,2025-02-01,Are we ready for another pandemic?,0.776435,0.771274,https://www.theguardian.com/global-development...
9,2025-09-03,‘The pandemic reinforced existing inequalities...,0.770412,0.297891,https://www.theguardian.com/world/2025/mar/09/...
